## vLLM-V0

vLLM-V0 版本包含:

1. `Continue_Batching.ipynb`
2. `vLLM-PageKVCache.ipynb`
3. `vLLM-PageAttention.ipynb`


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from typing import Dict, List, Set, Tuple, Optional, Any

torch.manual_seed(42)

## Config

In [2]:
from dataclasses import dataclass

@dataclass
class vLLMEngineConfig:
    max_batch_size = 4
    max_seq_len = 32
    max_prompt_len: int = 16

    # model & kv cache
    num_layers = 3

    # PageKV Cache Setting
    page_size = 64
    num_pages = 1024

    dim = 16
    num_heads = 2
    head_dim = 8
    vocab_size = 20

config = vLLMEngineConfig()
print(config.max_seq_len)

32


# Request

In [3]:
EOS_TOKEN = 0

In [4]:
class Request:
    def __init__(self,
                 request_id: int,
                 prompt: List[int],
                 max_len: int = 2048):
        self.request_id = request_id
        self.prompt = prompt
        self.generated_tokens = []
        self.status = "REQUEST_WAITING"  # WAITING, RUNNING, COMPLETED
        self.current_length = len(prompt)
        self.max_length = max_len

    def add_token(self, token: int):
        """添加生成的token到请求中"""
        self.generated_tokens.append(token)
        self.current_length += 1
        if self.is_finished():
            self.status = "REQUEST_COMPLETED"
            print(
                f'finished: ID.{self.request_id}, new_len:{len(self.generated_tokens)}')

    def is_finished(self) -> bool:
        """检查请求是否完成(达到最大长度或生成了EOS)"""
        result = (self.current_length >= self.max_length or (
            self.generated_tokens and self.generated_tokens[-1] == EOS_TOKEN))
        return result

    def get_full_sequence(self) -> List[int]:
        """获取完整的序列(prompt + 生成的tokens)"""
        return self.prompt + self.generated_tokens


In [5]:
req=Request(request_id=1, prompt=[1,2,3], max_len=10)
req.add_token(8)
req.get_full_sequence()

[1, 2, 3, 8]

## Schedular

即为之前的 RequestManager

In [6]:
from queue import deque

class Schedular:
    """管理所有请求的调度和状态"""

    def __init__(self, max_seq_len: int = 1024):
        self.max_seq_len = max_seq_len
        self.requests = {}  # request_id -> Request
        self.waiting_queue = deque()
        self.running_requests = set()

    def add_request(self, prompt: List[int], max_seq_len: int) -> int:
        """添加新请求，返回请求ID"""
        request_id = len(self.requests)
        request = Request(request_id, prompt, max_seq_len)
        self.requests[request_id] = request
        self.waiting_queue.append(request_id)
        return request_id

    def get_available_request(self) -> int:
        """获取可用的批次空位数量"""
        return len(self.waiting_queue)

    def get_pending_requests(self, max_count: int) -> List[Tuple[int, List[int]]]:
        """获取等待处理的请求"""
        available_slots = self.get_available_request()
        count = min(max_count, available_slots, len(self.waiting_queue))

        requests_to_process = []
        for _ in range(count):
            if not self.waiting_queue:
                break
            request_id = self.waiting_queue.popleft()
            request = self.requests[request_id]
            request.status = "REQUEST_RUNNING"
            self.running_requests.add(request_id)
            requests_to_process.append((request_id, request.prompt))
        return requests_to_process

    def update_request(self, request_id: int, next_token: int):
        """更新请求状态"""
        if request_id in self.requests:
            request = self.requests[request_id]
            request.add_token(next_token)
            if request.is_finished():
                self.running_requests.discard(request_id)

    def has_pending_requests(self) -> bool:
        """检查是否有未完成的请求"""
        return len(self.waiting_queue) > 0 or len(self.running_requests) > 0

    def get_num_pending_requests(self) -> int:
        return len(self.waiting_queue)

    def get_num_running_requests(self) -> int:
        return len(self.running_requests)

    def get_running_request_ids(self) -> List[int]:
        """获取当前正在运行的请求ID"""
        return list(self.running_requests)

In [7]:
requestor = Schedular(max_seq_len=config.max_seq_len)
print(requestor.add_request(prompt=[1, 2, 3], max_seq_len=config.max_seq_len))
print(requestor.has_pending_requests())
print(requestor.get_running_request_ids())

0
True
[]


## PageKVCache 设计

在 PageKVCache 进行分离：

1. 块表：记录索引，与外部业务无关
2. PageKVCacheEngine：申请存储一片巨大的空间，负责业务

## BlockTable

In [8]:
class BlockTable:
    """逻辑块表管理 - 仅负责分页资源管理"""

    def __init__(self, page_size: int, num_pages: int):
        self.page_size = page_size
        self.num_pages = num_pages
        self.free_pages = list(range(num_pages))
        self.allocated_pages = set()

        self.page_usage = [0] * num_pages  
        self.next_page = [-1] * num_pages  

    def _allocate_pages(self, num_pages: int, parent_block_id=-1):
        """分配指定数量的页"""
        if len(self.free_pages) < num_pages:
            return None

        allocated = self.free_pages[:num_pages]
        self.free_pages = self.free_pages[num_pages:]
        self.allocated_pages.update(allocated)

        # 初始化块状态
        for page_id in allocated:
            self.page_usage[page_id] = 0
            self.next_page[page_id] = -1

        if parent_block_id != -1:
            self.next_page[parent_block_id] = allocated[0]

        return allocated

    def _free_pages(self, page_ids: list[int]):
        """释放页"""
        for page_id in page_ids:
            if page_id in self.allocated_pages:
                self.allocated_pages.remove(page_id)
                self.free_pages.append(page_id)
                self.page_usage[page_id] = 0
                self.next_page[page_id] = -1

    def get_free_count(self) -> int:
        """获取空闲块数量"""
        return len(self.free_pages)

In [9]:
tabler = BlockTable(16,16)
tabler._allocate_pages(25)
tabler._allocate_pages(8)
tabler._allocate_pages(34)

tabler._free_pages([1,2])

## PageKVCacheEngine

In [10]:
class PageKVCacheEngine:
    """分页式管理 KV 缓存"""

    def __init__(self, config):
        # 初始化KV缓存 [layer, batch, seq, head, dim]

        self.num_pages = config.num_pages
        self.page_size = config.page_size
        self.num_layers = config.num_layers
        self.num_heads = config.num_heads

        self.block_table = BlockTable(self.page_size,
                                      self.num_pages,)

        # KV 的存储表由块表大小来管理
        self.k_cache = torch.zeros(config.num_layers,
                                   self.num_pages,
                                   self.page_size,
                                   config.num_heads,
                                   config.head_dim)
        self.v_cache = torch.zeros_like(self.k_cache)

        # 每个请求的长度
        self.sequence_lengths = {}

        # 请求与 block_id 的映射信息
        self.request_to_pages = {}  # request_id -> [page_id1, page_id2, ...]
        self.page_to_request = {}  # page_id -> request_id

    def has_active_requests(self) -> bool:
        """检查是否有活跃的请求"""
        return len(self.request_to_pages) > 0

    def has_available_pages(self, request_length) -> bool:
        """检查是否有可用的分页"""
        free_num_pages = self.block_table.get_free_count()
        return request_length < free_num_pages * self.page_size

    def allocate_request_pages(self, request_id, request_length) -> List[int]:
        """
        为 prefill 请求预分配页面, 对于正在解码的 decoding 请求，不需要预分配 page, 后续分配功能写在一起
        """

        allocate_pages_size = (request_length // self.page_size)+1
        allocate_pages_ids = self.block_table._allocate_pages(
            allocate_pages_size)
        if allocate_pages_ids == None:
            print(f'[ALLOCATE] request ID{request_id} pages faild')
            return []

        self.request_to_pages[request_id] = allocate_pages_ids

        for i in self.request_to_pages[request_id]:
            self.page_to_request[i] = request_id
        self.sequence_lengths[request_id] = 0

        print(
            f'[ALLOCATE] request ID{request_id} pages len {len(allocate_pages_ids)}')

        return allocate_pages_ids

    def free_request_pages(self, request_id: int):
        """释放请求占用的页面"""
        allocate_pages_ids = self.request_to_pages[request_id]
        self.block_table._free_pages([0, 1])

        self.k_cache[:, allocate_pages_ids, :, :, :] = 0
        self.v_cache[:, allocate_pages_ids, :, :, :] = 0

        del self.request_to_pages[request_id]
        for idx in allocate_pages_ids:
            del self.page_to_request[idx]
        print(
            f"[FREE] request ID{request_id} pages, len{len(allocate_pages_ids)}")

    def update_pages(self, request_id, new_kv_cache):
        # 1. Prefill: 填充到 requst_id -> pages 上
        # 2. Decoding: 找到 requst_id -> pages 上的最后一个 token, 如果最后一个块已满，需要重新申请一个新的块表。

        # 1. Update Decoding blocks
        # 对于 Decoding 存在增加一个新 token 导致要新加一个 block 的情况
        seq_len, _, _ = new_kv_cache[0][0].shape  # 0 层数据 K数据
        T = self.page_size

        if seq_len == 1:
            length = self.sequence_lengths[request_id]
            pages_ids = self.request_to_pages[request_id]
            if length % T == 0:
                new_block_id = self.block_table._allocate_pages(
                    1, pages_ids[-1])

                
                self.request_to_pages[request_id].append(new_block_id[0])
                self.page_to_request[new_block_id[0]] = request_id
            self.sequence_lengths[request_id] += 1
            cur_offset_len = self.sequence_lengths[request_id] % T
        else:
            # prefill 填充 context 长度
            self.sequence_lengths[request_id] = seq_len

        # 2. Update Decoding Stage KV-Cache
        if seq_len == 1:
            for i, layer_kv_cache in enumerate(new_kv_cache):
                # seq_len, num_heads, head_dim = layer_kv_cache[0].shape
                pages_ids = self.request_to_pages[request_id]
                cur_offset_len = self.sequence_lengths[request_id] % T

                # 最后一个块上加 Cache
                self.k_cache[i, pages_ids[-1], cur_offset_len,
                             :, :] = layer_kv_cache[0][0, :, :]
                self.v_cache[i, pages_ids[-1], cur_offset_len,
                             :, :] = layer_kv_cache[1][0, :, :]

        # 3. Update Prefill Stage KV-Cache
        # 更新简单
        else:
            for i, layer_kv_cache in enumerate(new_kv_cache):
                pages_ids = self.request_to_pages[request_id]
                for k, idx in enumerate(pages_ids):
                    self.k_cache[i, pages_ids, :, :, :] = layer_kv_cache[0]
                    self.v_cache[i, pages_ids, :, :, :] = layer_kv_cache[1]
                    

    def get_sequence_kvcache(self, request_ids: List[int]) -> Tuple[torch.Tensor, torch.Tensor]:
        pass

    def get_page_kvcache(self, request_ids: List[int]) -> Tuple[torch.Tensor, torch.Tensor]:
        """获取 page2batch 数据, request-wise"""
        L, _, T, H, D  = self.k_cache.shape

        N = 0 # num_pages
        batch_to_page = {}
        num_pages_len = []
        for t, idx in enumerate(request_ids):
            page_ids = self.request_to_pages[idx]
            num_pages_len.append(len(page_ids))
            for i, page_id in enumerate(page_ids):
                batch_to_page[i+N] = page_id
            N += len(page_ids)

        K = torch.zeros(L, N, T, H, D)
        V = torch.zeros(L, N, T, H, D)

        b_id = 0 # batch id start
        for _, idx in enumerate(request_ids):
            page_ids = self.request_to_pages[idx]
            num_pages = len(page_ids)
            
            K[:, b_id: b_id+num_pages, :, :, :] = self.k_cache[:,page_ids,:, :, :]
            V[:, b_id: b_id+num_pages, :, :, :] = self.v_cache[:,page_ids,:, :, :]

            b_id += num_pages

        return (K, V), num_pages_len, batch_to_page

    def get_request_info(self, ):

        for request_id, page_ids in self.request_to_pages.items():
            print(f'----Req.ID:{request_id}----')
            print('pages list:', page_ids)
            print('cur_length:', self.sequence_lengths[request_id])

In [11]:
cacher = PageKVCacheEngine(config)
print(cacher.k_cache.shape)
print(cacher.has_active_requests())

print('-' * 100)
cacher.allocate_request_pages(request_id=18, request_length=65)
print(cacher.get_request_info())

print('-' * 100)
cacher.allocate_request_pages(request_id=4, request_length=532)
print(cacher.get_request_info())

print('-' * 100)
cacher.free_request_pages(request_id=18)
print(cacher.get_request_info())

torch.Size([3, 1024, 64, 2, 8])
False
----------------------------------------------------------------------------------------------------
[ALLOCATE] request ID18 pages len 2
----Req.ID:18----
pages list: [0, 1]
cur_length: 0
None
----------------------------------------------------------------------------------------------------
[ALLOCATE] request ID4 pages len 9
----Req.ID:18----
pages list: [0, 1]
cur_length: 0
----Req.ID:4----
pages list: [2, 3, 4, 5, 6, 7, 8, 9, 10]
cur_length: 0
None
----------------------------------------------------------------------------------------------------
[FREE] request ID18 pages, len2
----Req.ID:4----
pages list: [2, 3, 4, 5, 6, 7, 8, 9, 10]
cur_length: 0
None


## Model

PageAttention 作为 Kernel, 不同阶段

1. Prefill: 输入是 `page_input_ids`,  输出是 `page_logits`, 要转换为 request-level 的 logits
2. Decoding: 输入是 `request_next_token`, `page_kv_cache`, 输出是 `request_logits`

In [12]:
def page_attention_prefill_kernel(Q, K, V, mask=None):
    """
    1 Request(batch_size=1), [Flash Attention-V2](https://zhuanlan.zhihu.com/p/670085985)
    Args
        Q: num_pages, num_heads, seq_len, head_dim (in decoding, seq_len=1)
        K: num_pages, num_heads, seq_len, head_dim
        V: num_pages, num_heads, seq_len, head_dim
        mask: num_pages, seq_len, seq_len
    Output
        O: num_pages, num_heads, seq_len, head_dim
    """
    
    N, H, T, D = Q.shape # batch_size, num_heads, seq_len, head_dim

    O_global = torch.zeros(N, H, T, D)
    for i in range(N): # Q Loop   
        O = torch.zeros(1, H, T, 1)
        M = torch.zeros(1, H, T, 1)
        L = torch.zeros(1, H, T, 1)
        Q_ = Q[i]
        for j in range(N): # KV Loop
            
            if j > i:
                continue
            K_, V_ = K[j], V[j]
            
            S_ij = Q_ @ K_.transpose(1,2) # num_heads, seq_len, seq_len
            M_ij, _ = torch.max(S_ij, dim = -1, keepdim=True) # num_heads, seq_len, 1
            M_new = torch.maximum(M_ij, M)
            P_ij = torch.exp(S_ij - M_new)
            L_ij = torch.sum(P_ij , dim = -1, keepdim=True ) # num_heads, seq_len, 1
            L_new = torch.exp(M - M_new) * L + L_ij
            O_i = torch.exp(M - M_new) * O + P_ij @ V_

            M = M_new
            L = L_new
            
        # re-scaled
        O_global[i] = (O_i / L_new).unsqueeze(dim = 0)
        
    return O_global


def page_attention_decoding_kernel(O, M, L, O_, M_, L_):
    """
    online softmax trick
    """
    O = torch.cat( [O, O_.unsqueeze(dim=0)], dim = 0)
    M = torch.cat( [M, M_.unsqueeze(dim=0)], dim = 0)
    L = torch.cat( [L, L_.unsqueeze(dim=0)], dim = 0)

    M_new,_ = torch.max(M, dim=0, keepdim=True)
    L_new =  torch.exp(M-M_new) * L
    L_new = torch.sum(L_new, dim=0, keepdim=True)
    O_new = (M-M_new) * (L/L_new) * O 
    
    O_new = torch.sum(O_new, keepdim=True, dim=0)
    
    return O_new

In [13]:
class PageAttentionBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.num_heads = config.num_heads
        self.dim = config.dim
        self.head_dim = config.head_dim
        self.WQ = nn.Linear(config.dim, config.dim, bias=False)
        self.WK = nn.Linear(config.dim, config.dim, bias=False)
        self.WV = nn.Linear(config.dim, config.dim, bias=False)
        self.WO = nn.Linear(config.dim, config.dim, bias=False)
        self.act = nn.ReLU()

    
    def forward_prefill(self,
                        X, 
                        attention_mask=None, # Attention mask 也是 page 型的 
                        request_num_pages : List[int] = [],
                        request_length=None,
                        KV_Cache  = [],
                        ):
        # Prefill
        B, T, _ = X.shape 
        H = self.num_heads
        D = self.head_dim
        
        Q, K, V = self.WQ(X), self.WK(X), self.WV(X)
        Q=Q.reshape(B, T, H, D).transpose(1,2)
        K=K.reshape(B, T, H, D).transpose(1,2)
        V=V.reshape(B, T, H, D).transpose(1,2)
        O = torch.zeros_like(Q)

        request_size = len(request_num_pages)
        offset = [0] * request_size
        for i in range(1, request_size):
            offset[i] = offset[i-1] + request_num_pages[i]
            
        for t in range(request_size): # Request Loop
            offset_i = offset[t]
            N = request_num_pages[t]
            Q_ = Q[offset_i: offset_i+N]
            K_ = K[offset_i: offset_i+N]
            V_ = V[offset_i: offset_i+N]
    
            O_ = page_attention_prefill_kernel(Q_, K_, V_, mask = attention_mask)
            O[offset_i: offset_i+N] = O_
    
        O = O.transpose(1,2).reshape(B, T, H*D)
        O = self.WO(O)
        O = X + self.act(O)

        return O, [K.transpose(1,2), V.transpose(1,2)]
    
    def forward_decoding(self,
                        X, 
                        attention_mask=None, # Attention mask 也是 page 型的 
                        request_num_pages : List[int] = [],
                        request_length=None,
                        KV_Cache  = None,
                        ):
        # Decoding
        B, T, _ = X.shape 
        H = self.num_heads
        D = self.head_dim
        
        # Proj
        q, k, v = self.WQ(X), self.WK(X), self.WV(X)
        q = q.reshape(B, 1, H, D).transpose(1,2)
        k = k.reshape(B, 1, H, D).transpose(1,2)
        v = v.reshape(B, 1, H, D).transpose(1,2)

        # TODO: Apply RoPE For Q,K

        # step1: init
        S = q @ k.transpose(2,3) # B, H, 1, 1)
        M_ = S.clone()
        L_ = torch.ones_like(M_)
        O_ = v

        # step2: repeat q (dispatch)
        repeat_tensor = torch.tensor(request_num_pages)
        q_ = torch.repeat_interleave(q, repeat_tensor, dim = 0) 

        # step3: block attenion, 
        K_, V_ = KV_Cache[0], KV_Cache[1] # bsz, seq_len, num_head, head_dim
        S = q_ @ K_.transpose(1,2).transpose(2,3)
        # TODO: Mask 
        M, _ = torch.max(S, dim=-1, keepdim=True)
        L = torch.sum( torch.exp( S - M), dim=-1, keepdim=True)
        P = torch.softmax(S, dim=-1)
        O = P @ V_.transpose(1,2)

        # step4: reudce result, (combine)
        offset = 0
        globle_O = torch.zeros_like(O_)
        for i, T in enumerate(request_num_pages):
            Oi = page_attention_decoding_kernel(
                O[offset:offset+T],
                M[offset:offset+T],
                L[offset:offset+T],
                O_[i],
                M_[i],
                L_[i],
            )
            globle_O[i] = Oi[0] # BH1D
            offset += T
            break
            
        O = globle_O.transpose(1,2).reshape(B, 1, H*D)
        O = self.WO(O)
        O = X + self.act(O)
        
        return O, [k.transpose(1,2), v.transpose(1,2)]

In [14]:
class PageToyModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.embd = nn.Embedding(config.vocab_size, config.dim)
        self.lm_head = nn.Linear(config.dim, config.vocab_size)
        self.decoder = nn.ModuleList(
            [PageAttentionBlock(config) for i in range(config.num_layers)]
        )

    def forward(self, x, kvcaches=None, current_length=None, request_num_pages=None):
        layer_kvcaches = []
        X = self.embd(x)

        for i, block in enumerate(self.decoder):
            if kvcaches == None:
                X, layer_kvcache = block.forward_prefill(X, request_num_pages=request_num_pages)
            else:
                X, layer_kvcache = block.forward_decoding(X,
                                         KV_Cache=[kvcaches[0][i],
                                                  kvcaches[1][i]],
                                         request_length=current_length,
                                                         request_num_pages=request_num_pages)

            layer_kvcaches.append(layer_kvcache)
        logits = self.lm_head(X)
        return logits, layer_kvcaches

## ModelWrapper

Prefill 处理技巧

1. 数据预处理：将数据由原来 batch-padding 模式，转化为 page-padding 模式
2. 输出：一个 Request 只有一个 next_token, Request 对应多个 pages, 取 pages 最后一页的模位置的 logits
3. update: 由于 page_input_ids 输入, 其 KV_cache 也是 Page-KV-Cache 形式的

Decoding 处理技巧

1. 数据预处理：(1) 整理 Request Page—CacheKV 排列顺序, 记录每个 request 的 pages, 便于后续做 block-attention 的结果 combine
2. 输出: 一个 Request 一个 q, logits 即是next-token-logits
3. update: 复用原 update 规则，KV 来源也异常简单。

In [15]:
class ModelWrapper:
    """封装模型的前向传播"""

    def __init__(self, model, kv_cache_manager: PageKVCacheEngine):
        self.model = model
        self.cacher = kv_cache_manager

    def prefill_requests(self, request_ids: List[int], prompts: List[List[int]]) -> Tuple[torch.Tensor, Any]:
        """
        预填充新请求
            1. 申请pages
            2. 组page级别的input_ids
        返回: request_page_ids:{[1,3,2], [5,4]}

        """
        if len(request_ids) == 0:
            return torch.tensor([])
        T = self.cacher.page_size

        # 数据预处理
        request_page_ids = []
        request_num_pages = []
        input_ids_list = []
        for request_id, prompt in zip(request_ids, prompts):

            # 获取 requst -> page_ids
            tmp_page_ids = self.cacher.allocate_request_pages(request_id,  len(prompt))
            tmp_len = len(tmp_page_ids)
            request_page_ids.append(tmp_page_ids)
            request_num_pages.append(tmp_len)

            # batch input_ids -> page input ids
            requst_input_ids = torch.zeros(len(tmp_page_ids), T, dtype=torch.long) # padding tensor
            for i in range(tmp_len):
                if i == tmp_len-1:
                    offset = len(prompt)%T
                    requst_input_ids[i, :offset] = torch.tensor(prompt[ i*T :  len(prompt)], dtype=torch.long)
                else:
                    requst_input_ids[i, :] = torch.tensor(prompt[ i*T : (i+1)*T ], dtype=torch.long)

                input_ids_list.append(requst_input_ids)
        input_ids = torch.cat(input_ids_list, dim = 0)
            
        # 执行预填充
        with torch.no_grad():
            logits, layer_kvcaches = self.model.forward(input_ids,
                                                        request_num_pages=request_num_pages,)
            

        # page logits 上取最后一个块的数据
        _, _, vocab_size = logits.shape
        page_logits = torch.zeros(len(request_ids), vocab_size)
        offset=0
        for i in range(len(request_ids)):
            
            batch_id = request_num_pages[i]-1
            idx = len(prompts[i]) % T
            page_logits[i] = logits[batch_id+offset, idx, :]
            offset+=request_num_pages[i] 
            
        return page_logits, layer_kvcaches, request_page_ids

    def decode_next_tokens(self, 
                           next_tokens: torch.Tensor, 
                           current_length=None,
                           num_pages_len=None,
                           KVCache=None) -> Tuple[torch.Tensor, Any]:
        """解码下一个token"""

        with torch.no_grad():
            logits, layer_kvcaches = self.model.forward(
                next_tokens,
                kvcaches=KVCache,
                request_num_pages=num_pages_len,
                current_length=current_length)

        return logits, layer_kvcaches

    def generate_next_tokens(self, logits: torch.Tensor) -> torch.Tensor:
        """从logits生成下一个token(贪婪采样)"""
        if len(logits) == 0:
            return torch.tensor([])
        return torch.argmax(logits, dim=-1)

## vLLM 类

In [19]:
class vLLMvOEngine:
    """vLLM 主引擎"""

    def __init__(self, model, config):
        self.cacher = PageKVCacheEngine(config)
        self.model_wrapper = ModelWrapper(model, self.cacher)
        self.schedular = Schedular(config.max_seq_len,)

    def add_request(self, prompt: List[int], max_seq_len) -> int:
        """添加新请求"""
        return self.schedular.add_request(prompt, max_seq_len)

    def step(self):
        """
        """
        request_ids = None
        layer_kvcaches = None
        # 阶段1: 处理解码(已有请求)
        if self.schedular.get_num_running_requests() > 0:
            
            request_ids = self.schedular.get_running_request_ids()


            # 准备输入token (上一个 step 生成的token)
            input_tokens = torch.tensor([
                self.schedular.requests[req_id].generated_tokens[-1]
                for req_id in request_ids
            ], dtype=torch.long)
            current_length = torch.tensor([
                self.schedular.requests[req_id].current_length
                for req_id in request_ids
            ], dtype=torch.long)
            input_tokens = input_tokens.unsqueeze(dim=1)

            # Page KVCache -> Batch KVCache
            # batch_kvcache = self.cacher.get_sequence_kvcache(request_ids)
            batch_kvcache, num_pages_len, batch_to_page = self.cacher.get_page_kvcache(request_ids)

            # 解码
            logits, layer_kvcaches = self.model_wrapper.decode_next_tokens(input_tokens,
                                                                           KVCache=batch_kvcache,
                                                                           num_pages_len=num_pages_len,
                                                                           current_length=current_length)
            next_tokens = self.model_wrapper.generate_next_tokens(logits)

            # update kv cache
            self.update_kvcache(request_ids, layer_kvcaches)

            # 更新状态
            for i, request_id in enumerate(request_ids):
                self.schedular.update_request(
                    request_id, next_tokens[i].item())
                if self.schedular.requests[request_id].is_finished():
                    self.cacher.free_request_pages(request_id)

        # 阶段2: 处理预填充(新请求)
        if self.schedular.get_num_pending_requests() > 0:
            pending_requests = self.schedular.get_pending_requests(
                config.num_pages
            )
            request_ids = [idx for idx, _ in pending_requests]
            prompts = [prompt for _, prompt in pending_requests]

            if pending_requests:
                logits, layer_kvcaches, request_page_ids = self.model_wrapper.prefill_requests(
                    request_ids, prompts)
                next_tokens = self.model_wrapper.generate_next_tokens(logits)
                for i, (request_id, _) in enumerate(pending_requests):
                    self.schedular.update_request(
                        request_id, 
                        next_tokens[i].item()
                    )

                self.update_kvcache(request_ids, layer_kvcaches, request_page_ids)

    def update_kvcache(self, request_ids, layer_kvcaches, request_page_ids=None):
        if request_ids != None:
            for i, idx in enumerate(request_ids):
                tmp_cache = [[layer_kvcache[0][i], layer_kvcache[1][i]]
                             for layer_kvcache in layer_kvcaches]
                self.cacher.update_pages(idx, tmp_cache)

    def has_pending_work(self) -> bool:
        """检查是否还有未完成的工作"""
        return self.schedular.has_pending_requests()

    def get_requests_info(self):
        pending = self.schedular.get_num_pending_requests()
        running = self.schedular.get_num_running_requests()
        total_request = len(self.schedular.requests)
        return pending, running, total_request

## 主函数

In [20]:
import random
# from random import randint
random.seed(42)  


def listen_request(config, p=0.01):
    prompt=[]
    prompt_len=0
    num = random.randint(1,100)
    if num/100.0 < p:
        prompt_len = random.randint(config.max_prompt_len//4, config.max_prompt_len)
        prompt=torch.randint(low=1, high=config.vocab_size, size=(1, prompt_len))
        prompt=prompt[0].tolist()
    return prompt, prompt_len

In [21]:
N = 32
count = 0

model = PageToyModel(config)
engine = vLLMvOEngine(model, config)

# main 
while 1:
    # 监听进程
    if count != N:
        prompt, prompt_len = listen_request(config, p=0.5)
        if prompt_len != 0:
            count += 1
            if count % (N//10) == 0:
                per = count / (N//10) 
                print('Running...:','*'*int(per),'-'*(10-int(per)))
            generate_len = random.randint(prompt_len, config.max_seq_len)
            engine.add_request(prompt, generate_len)
            pending, running, total = engine.get_requests_info()
            print(f'[Request Info] pending:{pending}/running:{running}/total:{total}')
            
    # 处理进程   
    engine.step()

    if not engine.has_pending_work() and count == N:
        print('process done')
        break

[Request Info] pending:1/running:0/total:1
[ALLOCATE] request ID0 pages len 1
[Request Info] pending:1/running:1/total:2
[ALLOCATE] request ID1 pages len 1
Running...: * ---------
[Request Info] pending:1/running:2/total:3
[ALLOCATE] request ID2 pages len 1
finished: ID.2, new_len:3
[FREE] request ID2 pages, len2
[Request Info] pending:1/running:2/total:4
[ALLOCATE] request ID3 pages len 1
[Request Info] pending:1/running:3/total:5
finished: ID.1, new_len:7
[FREE] request ID1 pages, len2
[ALLOCATE] request ID4 pages len 1
Running...: ** --------
[Request Info] pending:1/running:3/total:6
finished: ID.4, new_len:2
[FREE] request ID4 pages, len2
[ALLOCATE] request ID5 pages len 1
[Request Info] pending:1/running:3/total:7
[ALLOCATE] request ID6 pages len 1
finished: ID.6, new_len:6
[FREE] request ID6 pages, len2
[Request Info] pending:1/running:3/total:8
[ALLOCATE] request ID7 pages len 1
Running...: *** -------
[Request Info] pending:1/running:4/total:9
[ALLOCATE] request ID8 pages len 